In [0]:
# Silver: Feature Engineering for Churn Model
import pandas as pd
from sklearn.preprocessing import LabelEncoder

df_silver = spark.table("bronze_customer_churn").toPandas()

# Encode categorical variables
le_contract = LabelEncoder()
le_payment = LabelEncoder()
le_internet = LabelEncoder()
le_streaming = LabelEncoder()

df_silver['contract_type_encoded'] = le_contract.fit_transform(df_silver['contract_type'])
df_silver['payment_method_encoded'] = le_payment.fit_transform(df_silver['payment_method'])
df_silver['internet_service_encoded'] = le_internet.fit_transform(df_silver['internet_service'])
df_silver['has_streaming_encoded'] = le_streaming.fit_transform(df_silver['has_streaming'])

# Feature: average monthly spend per tenure month (engagement proxy)
df_silver['spend_per_tenure'] = df_silver['monthly_charges'] / df_silver['tenure_months']

# Data quality flag
df_silver['data_quality_flag'] = df_silver[['tenure_months', 'monthly_charges']].isna().any(axis=1)

print(f"Silver records: {len(df_silver)}")
print(f"Features added: contract_type_encoded, payment_method_encoded, internet_service_encoded, has_streaming_encoded, spend_per_tenure")
display(df_silver.head(10))

Silver records: 5000
Features added: contract_type_encoded, payment_method_encoded, internet_service_encoded, has_streaming_encoded, spend_per_tenure


customer_id,tenure_months,contract_type,monthly_charges,support_tickets,payment_method,internet_service,has_streaming,churned,contract_type_encoded,payment_method_encoded,internet_service_encoded,has_streaming_encoded,spend_per_tenure,data_quality_flag
CUST-02000,40,Two Year,66.61,0,Bank Transfer,Fiber Optic,Yes,0,2,0,1,1,1.66525,false
CUST-02001,42,Month-to-Month,88.67,1,Credit Card,DSL,Yes,0,0,1,0,1,2.111190476190476,false
CUST-02002,10,Month-to-Month,46.32,0,Credit Card,DSL,Yes,0,0,1,0,1,4.632,false
CUST-02003,19,Month-to-Month,43.84,0,Credit Card,No,No,0,0,1,2,0,2.3073684210526317,false
CUST-02004,6,One Year,95.92,1,Electronic Check,DSL,Yes,0,1,2,0,1,15.986666666666666,false
CUST-02005,32,Month-to-Month,53.43,3,Bank Transfer,DSL,No,0,0,0,0,0,1.6696875,false
CUST-02006,72,Two Year,33.42,3,Bank Transfer,Fiber Optic,No,0,2,0,1,0,0.46416666666666667,false
CUST-02007,10,Month-to-Month,52.87,2,Electronic Check,DSL,Yes,1,0,2,0,1,5.287,false
CUST-02008,5,Month-to-Month,56.75,0,Mailed Check,No,No,1,0,3,2,0,11.35,false
CUST-02009,14,One Year,103.87,4,Electronic Check,DSL,No,0,1,2,0,0,7.4192857142857145,false


In [0]:
# Write Silver table to Delta
spark.createDataFrame(df_silver).write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_customer_churn")

print(f"Silver table written: {spark.table('silver_customer_churn').count()} records")

Silver table written: 5000 records
